# Cell Graph Structure Exploration

Explore the topology of per-cell bipartite pixel graphs from MPX data.

Each cell has a bipartite graph: **A-pixels** (oligonucleotide-labeled spatial positions) connected to **B-pixels** (molecular interactions), with edges labeled by **protein marker**. We examine degree distributions, hub nodes, centrality, and how graph structure varies across cell types and experimental conditions.

In [ ]:
import sys
sys.path.insert(0, '/home/projects/nyosef/zvise/PixelGen/PixelGen')

import numpy as np
import pandas as pd
import networkx as nx
import igraph as ig
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from pixelator import read_pna
from tqdm import tqdm
from scipy import stats
from statsmodels.stats.multitest import multipletests

# --- Global figure style ---
sns.set_style("whitegrid")
sc.settings.set_figure_params(dpi=120, frameon=False, fontsize=12)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "font.family": "sans-serif",
})

RESULTS_DIR = Path("/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/results")
CACHE_DIR = Path("/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/cache")

COND_PALETTE = {"Mock": "#4c72b0", "Blinatumomab": "#dd8452"}
TIME_PALETTE = {"6h": "#55a868", "48h": "#c44e52"}

sample_meta = {
    "S001": {"time": "6h",  "condition": "Mock",           "target": "healthy B",  "tcells": "healthy T"},
    "S002": {"time": "6h",  "condition": "Blinatumomab",   "target": "healthy B",  "tcells": "healthy T"},
    "S003": {"time": "48h", "condition": "Mock",           "target": "healthy B",  "tcells": "healthy T"},
    "S004": {"time": "48h", "condition": "Blinatumomab",   "target": "healthy B",  "tcells": "healthy T"},
    "S005": {"time": "6h",  "condition": "Mock",           "target": "NALM-6",     "tcells": "healthy T"},
    "S006": {"time": "6h",  "condition": "Blinatumomab",   "target": "NALM-6",     "tcells": "healthy T"},
    "S007": {"time": "48h", "condition": "Mock",           "target": "NALM-6",     "tcells": "healthy T"},
    "S008": {"time": "48h", "condition": "Blinatumomab",   "target": "NALM-6",     "tcells": "healthy T"},
    "S009": {"time": "6h",  "condition": "Mock",           "target": "patient B",  "tcells": "patient T"},
    "S010": {"time": "6h",  "condition": "Blinatumomab",   "target": "patient B",  "tcells": "patient T"},
    "S011": {"time": "48h", "condition": "Mock",           "target": "patient B",  "tcells": "patient T"},
    "S012": {"time": "48h", "condition": "Blinatumomab",   "target": "patient B",  "tcells": "patient T"},
    "S013": {"time": "6h",  "condition": "Mock",           "target": "NALM-6",     "tcells": "patient T"},
    "S014": {"time": "6h",  "condition": "Blinatumomab",   "target": "NALM-6",     "tcells": "patient T"},
    "S016": {"time": "48h", "condition": "Blinatumomab",   "target": "NALM-6",     "tcells": "patient T"},
}

## 1. Load Data

Load pxl files for the edgelist and the cached annotated adata for cell type / condition labels.

In [ ]:
# Load annotated adata (cell types, QC filters already applied)
adata = sc.read_h5ad(CACHE_DIR / "adata_annotated.h5ad")
print(f"Annotated adata: {adata.shape}")
print(f"Cell types: {adata.obs['cell_type'].value_counts().to_dict()}")
print(f"Conditions: {adata.obs['condition'].value_counts().to_dict()}")

In [ ]:
# Verify pxl files exist (we load them per-sample later to avoid memory issues)
for sample_id in sample_meta:
    pxl_path = RESULTS_DIR / sample_id / "layout" / "layout" / f"{sample_id}.layout.pxl"
    if pxl_path.exists():
        print(f"  {sample_id}: {pxl_path.stat().st_size / 1e9:.1f} GB")
    else:
        print(f"  {sample_id}: NOT FOUND")

In [ ]:
# Inspect the edgelist structure on a SINGLE component (memory-safe)
from pixelator import read_pna
pg_single = read_pna([RESULTS_DIR / "S001" / "layout" / "layout" / "S001.layout.pxl"])

# Edgelist is a custom object — use .filter() then .to_df()
el_obj = pg_single.edgelist()
print(f"Edgelist type: {type(el_obj)}")
print(f"Components in S001: {len(el_obj.components)}")

# Get a single component's edgelist as DataFrame
one_comp = list(el_obj.components)[:1]
el_sample = pg_single.filter(components=one_comp).edgelist().to_df()
print(f"\nSingle cell edgelist shape: {el_sample.shape}")
print(f"Columns: {el_sample.columns.tolist()}")
print(f"\nUMI1 (pixel type 1) unique: {el_sample['umi1'].nunique()}")
print(f"UMI2 (pixel type 2) unique: {el_sample['umi2'].nunique()}")
print(f"UMI1/UMI2 overlap: {len(set(el_sample['umi1']) & set(el_sample['umi2']))}  (bipartite = 0)")
el_sample.head(10)

In [ ]:
# Stratified sample: ~50 cells per cell_type x condition
N_PER_GROUP = 50
np.random.seed(42)

subset_components = []
for (ct, cond), group_df in adata.obs.groupby(["cell_type", "condition"], observed=True):
    n_sample = min(N_PER_GROUP, len(group_df))
    sampled = group_df.sample(n=n_sample, random_state=42)
    subset_components.extend(sampled.index.tolist())
    print(f"  {ct} / {cond}: {n_sample} cells")

subset_components = sorted(set(subset_components))
print(f"\nTotal subset: {len(subset_components)} cells")

# Map each component to its sample for per-sample loading
comp_to_sample = adata.obs.loc[subset_components, 'sample'].to_dict()
sample_to_comps = {}
for comp, sample in comp_to_sample.items():
    sample_to_comps.setdefault(sample, []).append(comp)

for s, comps in sorted(sample_to_comps.items()):
    print(f"  {s}: {len(comps)} cells")

## 2. Compute Per-Cell Graph Metrics (Streaming)

Load one sample at a time, build graphs per cell, extract all metrics, discard the graph.  
We never hold more than one sample's edgelist in memory.

In [ ]:
from collections import Counter
import igraph as ig

def _gini(arr):
    """Gini coefficient via sorted-array formula. O(n log n), no large allocations."""
    arr = np.sort(arr)
    n = len(arr)
    if n == 0 or arr.sum() == 0:
        return 0.0
    index = np.arange(1, n + 1)
    return float((2 * np.sum(index * arr) - (n + 1) * np.sum(arr)) / (n * np.sum(arr)))

def compute_cell_metrics(comp_el):
    """Build bipartite graph, project, and compute all metrics for one cell.
    Uses igraph (C-based). ~1s per cell."""
    
    # --- Build bipartite graph via igraph ---
    u1_ids = comp_el['umi1'].unique()
    u2_ids = comp_el['umi2'].unique()
    n_u1, n_u2 = len(u1_ids), len(u2_ids)
    
    u1_map = {uid: i for i, uid in enumerate(u1_ids)}
    u2_map = {uid: i + n_u1 for i, uid in enumerate(u2_ids)}
    
    edge_agg = comp_el.groupby(['umi1', 'umi2']).size().reset_index(name='weight')
    sources = edge_agg['umi1'].map(u1_map).values
    targets = edge_agg['umi2'].map(u2_map).values
    
    G = ig.Graph(n=n_u1 + n_u2, edges=list(zip(sources, targets)), directed=False)
    G.vs['type'] = [False] * n_u1 + [True] * n_u2
    G.es['weight'] = edge_agg['weight'].values.tolist()
    
    n_edges_bip = G.ecount()
    
    # Bipartite degree stats
    u1_degrees = np.array(G.degree(range(n_u1)))
    u2_degrees = np.array(G.degree(range(n_u1, n_u1 + n_u2)))
    
    result = {
        'n_umi1_pixels': n_u1,
        'n_umi2_pixels': n_u2,
        'n_edges_bip': n_edges_bip,
        'density': n_edges_bip / (n_u1 * n_u2) if (n_u1 * n_u2) > 0 else 0,
        'u1_u2_ratio': n_u1 / n_u2 if n_u2 > 0 else np.nan,
        'u1_degree_mean': u1_degrees.mean(),
        'u1_degree_median': np.median(u1_degrees),
        'u1_degree_max': int(u1_degrees.max()),
        'u1_degree_std': u1_degrees.std(),
        'u1_degree_skew': float(stats.skew(u1_degrees)) if len(u1_degrees) > 2 else np.nan,
        'u2_degree_mean': u2_degrees.mean(),
        'u2_degree_median': np.median(u2_degrees),
        'u2_degree_max': int(u2_degrees.max()),
        'u2_degree_std': u2_degrees.std(),
        'u2_degree_skew': float(stats.skew(u2_degrees)) if len(u2_degrees) > 2 else np.nan,
    }
    
    # --- Projected graph (UMI1-UMI1 via shared UMI2) ---
    if n_u1 >= 2:
        G_proj = G.bipartite_projection(which=0)
        proj_degrees = np.array(G_proj.degree())
        n_proj = G_proj.vcount()
        
        cc = G_proj.connected_components()
        largest_cc_size = max(cc.sizes())
        
        result.update({
            'proj_n_edges': G_proj.ecount(),
            'proj_density': G_proj.density(),
            'proj_degree_mean': proj_degrees.mean(),
            'proj_degree_median': np.median(proj_degrees),
            'proj_degree_max': int(proj_degrees.max()),
            'proj_degree_std': proj_degrees.std(),
            'n_connected_components': len(cc),
            'largest_cc_frac': largest_cc_size / n_u1,
            'avg_clustering': G_proj.transitivity_avglocal_undirected(mode="zero"),
            'transitivity': G_proj.transitivity_undirected(),
        })
        
        # Path length only for small graphs (< 1000 nodes), too slow otherwise
        if largest_cc_size > 1 and largest_cc_size <= 1000:
            largest_cc_idx = cc.sizes().index(largest_cc_size)
            sub_ids = [v for v, m in enumerate(cc.membership) if m == largest_cc_idx]
            G_sub = G_proj.subgraph(sub_ids)
            result['avg_path_length'] = G_sub.average_path_length()
            result['diameter'] = G_sub.diameter()
        
        # --- Degree centrality (O(V), instant) ---
        dc = proj_degrees / (n_proj - 1) if n_proj > 1 else np.zeros(n_proj)
        
        result.update({
            'dc_mean': dc.mean(), 'dc_max': dc.max(),
            'dc_gini': _gini(dc),
        })
        
        # Hub marker enrichment: markers on top-10% degree UMI1 pixels vs rest
        threshold = np.percentile(dc, 90)
        hub_pixel_indices = set(np.where(dc >= threshold)[0])
        hub_umis = set(u1_ids[i] for i in hub_pixel_indices)
        nonhub_umis = set(u1_ids[i] for i in range(n_u1) if i not in hub_pixel_indices)
        
        hub_markers = Counter(comp_el[comp_el['umi1'].isin(hub_umis)]['marker_1'])
        nonhub_markers = Counter(comp_el[comp_el['umi1'].isin(nonhub_umis)]['marker_1'])
        result['_hub_markers'] = dict(hub_markers)
        result['_nonhub_markers'] = dict(nonhub_markers)
        
        del G_proj
    
    # --- Per-marker stats ---
    marker_counts = comp_el.groupby('marker_1').agg(
        n_edges=('umi1', 'size'),
        n_pixels=('umi1', 'nunique'),
    )
    marker_counts['edges_per_pixel'] = marker_counts['n_edges'] / marker_counts['n_pixels']
    result['_marker_stats'] = marker_counts.to_dict('index')
    
    result['_u1_degrees'] = u1_degrees.tolist()
    result['_u2_degrees'] = u2_degrees.tolist()
    
    del G
    return result

In [ ]:
METRICS_CACHE = CACHE_DIR / "graph_structure_metrics.parquet"
EXTRAS_CACHE = CACHE_DIR / "graph_structure_extras.pkl"

if METRICS_CACHE.exists() and EXTRAS_CACHE.exists():
    print("Loading cached metrics...")
    df_all = pd.read_parquet(METRICS_CACHE)
    if 'component' in df_all.columns:
        df_all = df_all.set_index('component')
    import pickle
    with open(EXTRAS_CACHE, 'rb') as f:
        extras = pickle.load(f)
    all_u1_degrees = extras['all_u1_degrees']
    all_u2_degrees = extras['all_u2_degrees']
    degree_by_cell = extras['degree_by_cell']
    hub_markers_agg = extras['hub_markers_agg']
    nonhub_markers_agg = extras['nonhub_markers_agg']
    marker_stats_list = extras['marker_stats_list']
    print(f"Loaded metrics for {len(df_all)} cells")
else:
    all_results = {}
    all_u1_degrees = []
    all_u2_degrees = []
    degree_by_cell = []
    hub_markers_agg = Counter()
    nonhub_markers_agg = Counter()
    marker_stats_list = []
    
    total_cells = sum(len(c) for c in sample_to_comps.values())
    pbar = tqdm(total=total_cells, desc="Processing cells")

    for sample_id in sorted(sample_to_comps.keys()):
        comps_in_sample = sample_to_comps[sample_id]
        pxl_path = RESULTS_DIR / sample_id / "layout" / "layout" / f"{sample_id}.layout.pxl"
        
        pbar.set_postfix(sample=sample_id, status="opening...")
        pg_s = read_pna([pxl_path])
        
        # Process each cell individually — pg.filter is fast (~0.2s per cell)
        # No need to load the entire sample edgelist
        for comp in comps_in_sample:
            pbar.set_postfix(sample=sample_id, cell=comp[:8])
            
            comp_el = pg_s.filter(components=[comp]).edgelist().to_df()
            if len(comp_el) == 0:
                pbar.update(1)
                continue
            
            result = compute_cell_metrics(comp_el)
            
            meta = adata.obs.loc[comp]
            ct = meta.get('cell_type', 'Unknown')
            cond = meta.get('condition', 'Unknown')
            
            u1d = result.pop('_u1_degrees')
            u2d = result.pop('_u2_degrees')
            all_u1_degrees.extend(u1d)
            all_u2_degrees.extend(u2d)
            for d in u1d:
                degree_by_cell.append((comp, 'umi1', d, ct, cond))
            for d in u2d:
                degree_by_cell.append((comp, 'umi2', d, ct, cond))
            
            hm = result.pop('_hub_markers', {})
            nhm = result.pop('_nonhub_markers', {})
            hub_markers_agg.update(hm)
            nonhub_markers_agg.update(nhm)
            
            ms = result.pop('_marker_stats', {})
            for marker, mstats in ms.items():
                marker_stats_list.append({
                    'component': comp, 'marker': marker, 'cell_type': ct, 'condition': cond,
                    **mstats
                })
            
            all_results[comp] = result
            pbar.update(1)
        
        del pg_s
    
    pbar.close()

    df_all = pd.DataFrame.from_dict(all_results, orient='index')
    df_all.index.name = 'component'
    
    obs_cols = ['cell_type', 'condition', 'time', 'sample', 'cell_system', 'n_umi', 'n_edges']
    obs_cols = [c for c in obs_cols if c in adata.obs.columns]
    df_all = df_all.join(adata.obs[obs_cols])
    
    df_all.to_parquet(METRICS_CACHE)
    import pickle
    with open(EXTRAS_CACHE, 'wb') as f:
        pickle.dump({
            'all_u1_degrees': all_u1_degrees,
            'all_u2_degrees': all_u2_degrees,
            'degree_by_cell': degree_by_cell,
            'hub_markers_agg': hub_markers_agg,
            'nonhub_markers_agg': nonhub_markers_agg,
            'marker_stats_list': marker_stats_list,
        }, f)
    print(f"\nComputed and cached metrics for {len(df_all)} cells")

print(f"df_all shape: {df_all.shape}")
df_all.describe().round(2)

## 3. Degree Distribution Analysis

In [ ]:
# Build degree DataFrame from collected tuples
df_degrees = pd.DataFrame(degree_by_cell, columns=['component', 'pixel_type', 'degree', 'cell_type', 'condition'])
print(f"Total pixel-level records: {len(df_degrees)}")
print(f"UMI1 pixels: {len(all_u1_degrees)}, UMI2 pixels: {len(all_u2_degrees)}")

cell_types = sorted(df_all['cell_type'].dropna().unique())
ct_palette = dict(zip(cell_types, sns.color_palette("Set2", len(cell_types))))

In [ ]:
# Aggregate degree distribution across all sampled cells
all_degrees = np.array(all_u1_degrees + all_u2_degrees)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(all_degrees, bins=100, color='#4c72b0', alpha=0.7, density=True)
axes[0].set_xlabel('Degree')
axes[0].set_ylabel('Density')
axes[0].set_title('Degree Distribution')

# Log-log CCDF (power-law check) — subsample for speed
rng = np.random.default_rng(42)
sub = rng.choice(all_degrees, size=min(50000, len(all_degrees)), replace=False)
sorted_d = np.sort(sub)[::-1]
ccdf = np.arange(1, len(sorted_d) + 1) / len(sorted_d)
axes[1].loglog(sorted_d, ccdf, '.', alpha=0.3, color='#4c72b0', markersize=1)
axes[1].set_xlabel('Degree (log)')
axes[1].set_ylabel('P(X >= x)')
axes[1].set_title('CCDF (power-law check)')

plt.tight_layout()
plt.show()
print(f"Total pixels: {len(all_degrees):,}, median degree: {np.median(all_degrees):.0f}, "
      f"mean: {np.mean(all_degrees):.1f}, max: {np.max(all_degrees)}")

In [ ]:
# Per-cell mean degree by cell type
cell_types = sorted(df_all['cell_type'].dropna().unique())
ct_palette = dict(zip(cell_types, sns.color_palette("Set2", len(cell_types))))

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df_all, x='cell_type', y='u1_degree_mean', palette=ct_palette, 
            fliersize=2, ax=ax, order=cell_types)
ax.set_xlabel('')
ax.set_ylabel('Mean Degree')
ax.set_title('Mean Node Degree by Cell Type')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Per-cell mean degree by condition
fig, ax = plt.subplots(figsize=(5, 4))
sns.boxplot(data=df_all, x='condition', y='u1_degree_mean', palette=COND_PALETTE,
            fliersize=2, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('Mean Degree')
ax.set_title('Mean Node Degree by Condition')
plt.tight_layout()
plt.show()

In [ ]:
# Per-cell degree stats by cell type
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (metric, title) in zip(axes, [
    ('u1_degree_mean', 'Mean Degree'),
    ('u1_degree_max', 'Max Degree'),
    ('u1_degree_skew', 'Degree Skewness'),
]):
    sns.boxplot(data=df_all, x='cell_type', y=metric,
                palette=ct_palette, fliersize=2, ax=ax, order=cell_types)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 4. Projected Graph Topology

The bipartite graph is projected onto UMI1 pixels (connected if they share a UMI2 neighbor). Metrics like clustering coefficient, transitivity, and path length are already computed in `df_all`.

In [ ]:
# Projected graph topology by cell type
proj_metrics = ['proj_degree_mean', 'avg_clustering', 'transitivity',
                'avg_path_length', 'largest_cc_frac', 'proj_density']
proj_titles = ['Mean Degree (projected)', 'Avg Clustering Coeff', 'Transitivity',
               'Avg Shortest Path Length', 'Fraction in Largest CC', 'Graph Density']
proj_metrics = [m for m in proj_metrics if m in df_all.columns]
proj_titles = proj_titles[:len(proj_metrics)]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, metric, title in zip(axes.flat, proj_metrics, proj_titles):
    sns.violinplot(data=df_all, x='cell_type', y=metric,
                   palette=ct_palette, inner='box', ax=ax, linewidth=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Projected Graph Topology by Cell Type', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Same metrics by condition
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, metric, title in zip(axes.flat, proj_metrics, proj_titles):
    sns.violinplot(data=df_all, x='condition', y=metric,
                   palette=COND_PALETTE, inner='box', ax=ax, linewidth=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Projected Graph Topology by Condition', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Hub Node Analysis

Identify hub A-pixels using centrality measures on the projected graph. Examine which protein markers are enriched at hub positions.

In [ ]:
# Centrality summary distributions (per-cell level from df_all)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cent_metrics = [('dc_mean', 'Mean Degree Centrality'), 
                ('dc_max', 'Max Degree Centrality'),
                ('dc_gini', 'Degree Centrality Gini\n(inequality of connectivity)')]

for ax, (metric, title) in zip(axes, cent_metrics):
    if metric not in df_all.columns:
        ax.set_visible(False)
        continue
    sns.violinplot(data=df_all, x='cell_type', y=metric,
                   palette=ct_palette, inner='box', ax=ax, linewidth=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Centrality Metrics by Cell Type', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Marker enrichment at hub positions (aggregated across all cells)
hub_total = sum(hub_markers_agg.values())
nonhub_total = sum(nonhub_markers_agg.values())

all_markers_set = set(hub_markers_agg.keys()) | set(nonhub_markers_agg.keys())
marker_enrichment = []
for m in all_markers_set:
    hub_freq = hub_markers_agg.get(m, 0) / hub_total if hub_total > 0 else 0
    nonhub_freq = nonhub_markers_agg.get(m, 0) / nonhub_total if nonhub_total > 0 else 0
    log2_fc = np.log2((hub_freq + 1e-6) / (nonhub_freq + 1e-6))
    marker_enrichment.append({'marker': m, 'hub_freq': hub_freq, 'nonhub_freq': nonhub_freq, 'log2_fc': log2_fc})

df_enrichment = pd.DataFrame(marker_enrichment).sort_values('log2_fc', ascending=False)

n_show = 15
top_enriched = pd.concat([df_enrichment.head(n_show), df_enrichment.tail(n_show)])

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#dd8452' if x > 0 else '#4c72b0' for x in top_enriched['log2_fc']]
ax.barh(top_enriched['marker'], top_enriched['log2_fc'], color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel('log2(hub freq / non-hub freq)')
ax.set_title('Marker Enrichment at Hub Pixels (top 10% degree centrality)', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Marker-Specific Graph Properties

For each marker, extract the subgraph induced by edges carrying that marker and compute graph metrics. This reveals which markers form spatially concentrated vs dispersed patterns.

In [ ]:
# Build marker stats DataFrame from collected records
df_marker_stats = pd.DataFrame(marker_stats_list)

# Compute fraction of edges per marker within each cell
total_per_cell = df_marker_stats.groupby('component')['n_edges'].transform('sum')
df_marker_stats['frac_edges'] = df_marker_stats['n_edges'] / total_per_cell

print(f"Marker stats records: {len(df_marker_stats)}")
print(f"Unique markers: {df_marker_stats['marker'].nunique()}")
df_marker_stats.head()

In [ ]:
# Aggregate per-marker stats
marker_agg = df_marker_stats.groupby('marker').agg(
    mean_edges=('n_edges', 'mean'),
    mean_frac_edges=('frac_edges', 'mean'),
    mean_n_pixels=('n_pixels', 'mean'),
    mean_edges_per_pixel=('edges_per_pixel', 'mean'),
).sort_values('mean_edges', ascending=False)

top_markers = marker_agg.head(40)

fig, axes = plt.subplots(1, 2, figsize=(10, 8))

axes[0].barh(top_markers.index[::-1], top_markers['mean_edges'][::-1], color='#4c72b0', alpha=0.8)
axes[0].set_xlabel('Mean Edges per Cell')
axes[0].set_title('Top 40 Markers by Edge Count')

axes[1].barh(top_markers.index[::-1], top_markers['mean_frac_edges'][::-1], color='#dd8452', alpha=0.8)
axes[1].set_xlabel('Mean Fraction of Total Edges')
axes[1].set_title('Top 40 Markers by Edge Fraction')

plt.tight_layout()
plt.show()

In [ ]:
# Marker graph properties by cell type: edges_per_pixel (spatial concentration)
top_10_markers = marker_agg.head(10).index.tolist()

fig, axes = plt.subplots(2, 5, figsize=(25, 8))
for ax, marker in zip(axes.flat, top_10_markers):
    sub = df_marker_stats[df_marker_stats['marker'] == marker]
    sns.boxplot(data=sub, x='cell_type', y='edges_per_pixel', 
                palette=ct_palette, ax=ax, linewidth=0.5)
    ax.set_title(marker, fontweight='bold', fontsize=11)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Edges per Pixel by Cell Type (top 10 markers)\nHigher = more concentrated spatially',
             fontsize=14, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

## 7. Graph Metrics vs Cell Phenotype

Examine how graph topology metrics relate to known cell properties (cell type, condition, UMI counts). Overlay graph metrics on the existing scVI UMAP to see if they capture spatial structure beyond abundance.

In [ ]:
# Correlation of graph metrics with cell size (n_umi, n_edges from QC)
size_metrics = ['n_umi', 'n_edges']
size_metrics = [m for m in size_metrics if m in df_all.columns]
graph_metrics = ['n_umi1_pixels', 'n_umi2_pixels', 'density', 'u1_degree_mean', 'u1_degree_skew',
                 'proj_degree_mean', 'avg_clustering', 'transitivity', 'avg_path_length',
                 'dc_gini', 'dc_max']
graph_metrics = [m for m in graph_metrics if m in df_all.columns]

corr_records = []
for sm in size_metrics:
    for gm in graph_metrics:
        valid = df_all[[sm, gm]].dropna()
        if len(valid) < 10:
            continue
        r, p = stats.spearmanr(valid[sm], valid[gm])
        corr_records.append({'size_metric': sm, 'graph_metric': gm, 'spearman_r': r, 'p_value': p})

df_corr = pd.DataFrame(corr_records)
corr_pivot = df_corr.pivot(index='graph_metric', columns='size_metric', values='spearman_r')

fig, ax = plt.subplots(figsize=(6, 8))
sns.heatmap(corr_pivot, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Graph Metrics vs Cell Size\n(Spearman correlation)', fontweight='bold')
plt.tight_layout()
plt.show()

print("\nMetrics with |r| < 0.3 to n_umi (potentially independent of cell size):")
low_corr = df_corr[(df_corr['size_metric'] == 'n_umi') & (df_corr['spearman_r'].abs() < 0.3)]
for _, row in low_corr.iterrows():
    print(f"  {row['graph_metric']}: r={row['spearman_r']:.3f}")

In [ ]:
# Overlay graph metrics on UMAP (if UMAP coordinates available in adata)
graph_cols_to_add = ['density', 'u1_degree_mean', 'u1_degree_skew', 
                     'avg_clustering', 'transitivity', 'avg_path_length']
graph_cols_to_add = [c for c in graph_cols_to_add if c in df_all.columns]

adata_subset = adata[adata.obs_names.isin(df_all.index)].copy()
for col in graph_cols_to_add:
    adata_subset.obs[col] = df_all.loc[adata_subset.obs_names, col].values

if 'X_umap' in adata_subset.obsm:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax, col in zip(axes.flat, graph_cols_to_add):
        sc.pl.umap(adata_subset, color=col, ax=ax, show=False, 
                   frameon=False, size=40, color_map='viridis', vmax='p95')
        ax.set_title(col, fontweight='bold')
    plt.suptitle('Graph Topology Metrics on scVI UMAP', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No UMAP coordinates found in adata. Skipping UMAP overlay.")

## 8. Graph Comparison Across Conditions

Statistical tests comparing graph metric distributions between Mock vs Blinatumomab within each cell type.

In [ ]:
# Mann-Whitney U tests: Mock vs Blinatumomab per cell type per metric
test_metrics = ['density', 'u1_degree_mean', 'u1_degree_skew', 'u1_u2_ratio',
                'proj_degree_mean', 'avg_clustering', 'transitivity', 'avg_path_length']
test_metrics = [m for m in test_metrics if m in df_all.columns]

test_results = []
for ct in df_all['cell_type'].dropna().unique():
    for metric in test_metrics:
        mock = df_all[(df_all['cell_type'] == ct) & (df_all['condition'] == 'Mock')][metric].dropna()
        blina = df_all[(df_all['cell_type'] == ct) & (df_all['condition'] == 'Blinatumomab')][metric].dropna()
        
        if len(mock) < 3 or len(blina) < 3:
            continue
        
        u_stat, p_val = stats.mannwhitneyu(mock, blina, alternative='two-sided')
        n1, n2 = len(mock), len(blina)
        effect_size = 1 - (2 * u_stat) / (n1 * n2)
        
        test_results.append({
            'cell_type': ct,
            'metric': metric,
            'mock_median': mock.median(),
            'blina_median': blina.median(),
            'u_stat': u_stat,
            'p_value': p_val,
            'effect_size': effect_size,
            'n_mock': n1,
            'n_blina': n2,
        })

df_tests = pd.DataFrame(test_results)

if len(df_tests) > 0:
    _, df_tests['p_adj'], _, _ = multipletests(df_tests['p_value'], method='fdr_bh')
    df_tests['sig'] = df_tests['p_adj'].apply(lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns')
    
    print("Significant results (p_adj < 0.05):")
    sig = df_tests[df_tests['p_adj'] < 0.05].sort_values('p_adj')
    if len(sig) > 0:
        print(sig[['cell_type', 'metric', 'mock_median', 'blina_median', 'effect_size', 'p_adj', 'sig']].to_string(index=False))
    else:
        print("  No significant differences found")

df_tests

In [ ]:
# Summary heatmap: effect size per cell_type x metric
if len(df_tests) > 0:
    effect_pivot = df_tests.pivot(index='metric', columns='cell_type', values='effect_size')
    sig_pivot = df_tests.pivot(index='metric', columns='cell_type', values='sig')
    
    # Build annotation strings
    annot = effect_pivot.round(2).astype(str)
    for col in annot.columns:
        for idx in annot.index:
            if idx in sig_pivot.index and col in sig_pivot.columns:
                s = sig_pivot.loc[idx, col]
                if pd.notna(s) and s != 'ns':
                    annot.loc[idx, col] = annot.loc[idx, col] + s
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(effect_pivot, annot=annot, fmt='', cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, ax=ax, linewidths=0.5)
    ax.set_title('Effect Size: Mock vs Blinatumomab\n(rank-biserial r, * = FDR < 0.05)',
                 fontweight='bold')
    ax.set_ylabel('Graph Metric')
    ax.set_xlabel('Cell Type')
    plt.tight_layout()
    plt.show()